In [59]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import json
#DataStore store stores the samples and their corresponding labels
#DataLoader wraps an iterable around the Dataset to enable easy access to the samples.


In [60]:

"""
for i, (session_id, eventstotal) in enumerate(objects):
    print(f"Session :", session_id)
    print(f"event number :", eventstotal)
    
    for singular_event in eventstotal:
        print(datetime.datetime.fromtimestamp(int(singular_event['ts'] / 1000)).strftime('%d-%m-%Y %H:%M:%S'))S
    if i == 1:
        break
        """

'\nfor i, (session_id, eventstotal) in enumerate(objects):\n    print(f"Session :", session_id)\n    print(f"event number :", eventstotal)\n    \n    for singular_event in eventstotal:\n        print(datetime.datetime.fromtimestamp(int(singular_event[\'ts\'] / 1000)).strftime(\'%d-%m-%Y %H:%M:%S\'))S\n    if i == 1:\n        break\n        '

In [61]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))


<class 'generator'>


In [62]:
session = []

for i, (session_id, eventstotal) in enumerate(extract_json("train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    session.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i == 2:
        break


In [63]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)


    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [ ]:
dataset = OttoDataSetSession(session)

for i in range(0,2):
    sample = dataset[i]
    print(sample)